# SNA Experiment 3 – Modeling a network
**Runtime:** Runtime → Change runtime type → **R**. Then Runtime → Run all.

Age is approximate: `2022 − birthyear` (the dataset is a 2022 snapshot of Congress).
Number of social media accounts = how many of twitter / facebook / youtube a member has (0–3).

In [ ]:
library(dplyr)
library(ggplot2)

load(url("https://dssoc.github.io/datasets/congress.RData"))

## Prepare one table: age + number of social media accounts

In [ ]:
members <- congress %>%
  inner_join(congress_contact, by = "bioguide_id") %>%
  mutate(
    age = 2022 - birthyear,
    has_twitter  = !is.na(twitter)  & twitter  != "",
    has_facebook = !is.na(facebook) & facebook != "",
    has_youtube  = !is.na(youtube)  & youtube  != "",
    n_accounts = has_twitter + has_facebook + has_youtube
  )

table(members$n_accounts)

## (a) Box plot of approximate age by gender

In [ ]:
members %>%
  group_by(gender) %>%
  summarise(avg_age = mean(age), members = n())

ggplot(members, aes(x = gender, y = age)) +
  geom_boxplot() +
  labs(title = "Age of Congress members by gender", x = "Gender", y = "Approximate age")

## (b) Scatter plot: age vs number of social media accounts, with a linear trendline

In [ ]:
ggplot(members, aes(x = age, y = n_accounts)) +
  geom_jitter(width = 0, height = 0.1, alpha = 0.4) +
  geom_smooth(method = "lm", formula = y ~ x) +
  labs(title = "Age vs number of social media accounts",
       x = "Approximate age", y = "Number of accounts")

## (c) Linear regression: n_accounts ~ type + age + party + gender

In [ ]:
model <- lm(n_accounts ~ type + age + party + gender, data = members)
summary(model)

In [ ]:
age_coef <- coef(model)["age"]
p_value  <- summary(model)$coefficients["age", "Pr(>|t|)"]

cat("Age coefficient:", round(age_coef, 4), "  p-value:", round(p_value, 4), "\n")
cat("Interpretation: holding type, party and gender constant, each extra year of age changes the\n",
    "expected number of social media accounts by", round(age_coef, 4),
    "(about", round(age_coef * 10, 3), "per 10 years).\n")